In [ ]:
import  osiris_utils as ou
from matplotlib import pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter


In [ ]:
def createSimDic(path, sim_labels, test):
    sim = {}
    for key in sim_labels.keys():
        sim[key] = {}
        for dtw in sim_labels[key]:
            sim[key][dtw] = ou.Simulation(f"{path}/{test}/{key}/dtw{dtw}/{key}.in")
    return sim

In [ ]:
# Normalize axis to w_ce


def _set_scaled_formatter(axis, scale_factor, fmt=".2f"):
    axis.set_major_formatter(
        FuncFormatter(lambda v, pos: f"{v*scale_factor:{fmt}}")
    )

def scale_x_ax(scale_factor, fig, ax, label=r"$t[1 / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_xlabel(label)
    if lock_ticks:
        # freeze current tick positions
        ticks = ax.get_xticks()
        ax.xaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    return fig, ax

def scale_y_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_ylabel(label)
    if lock_ticks:
        ticks = ax.get_yticks()
        ax.yaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    return fig, ax

def scale_z_ax(scale_factor, fig, ax, label=r"$x[c / \Omega_p]$", lock_ticks=False, fmt=".2f"):
    ax.set_zlabel(label)
    if lock_ticks:
        ticks = ax.get_zticks()
        ax.zaxis.set_major_locator(FixedLocator(ticks))
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax

def scale_3d_axes(scale_factor, fig, ax, fmt=".2f"):
    # ax.set_xlabel(r"$x_1[c / \Omega_e]$")
    # ax.set_ylabel(r"$x_2[c / \Omega_e]$")
    # ax.set_zlabel(r"$x_3[c / \Omega_e]$")
    _set_scaled_formatter(ax.xaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.yaxis, scale_factor, fmt)
    _set_scaled_formatter(ax.zaxis, scale_factor, fmt)
    return fig, ax


In [ ]:
class curvDriftTheo:
    def __init__(self, sim, B=None, rqm = -1, direc = -1):
        self.rqm = rqm
        self.direc = direc
        self.sim = sim
        self.x1_0 = sim["test_electrons"]["tracks"]["x1"][:,0]
        self.x2_0 = sim["test_electrons"]["tracks"]["x2"][:,0]
        self.x3_0 = sim["test_electrons"]["tracks"]["x3"][:,0]
        self.phi0 = np.arctan2(self.x2_0, self.x1_0)
        
        self.R0 = np.sqrt(sim["test_electrons"]["tracks"]["x1"][:,0]**2 + sim["test_electrons"]["tracks"]["x2"][:,0]**2)

        if B is not None:
            self.B0 = B
        else:
            self.B0 = np.sqrt(sim["test_electrons"]["tracks"]["B1"][:,0]**2 + sim["test_electrons"]["tracks"]["B2"][:,0]**2 + sim["test_electrons"]["tracks"]["B3"][:,0]**2)

        self.vc, self.v_par = self._curv_v()

    def b1(self, x1, x2, x3):
        return self.B0 * (-x2) / np.sqrt(x1**2 + x2**2)
    def b2(self, x1, x2, x3):
        return self.B0 * x1 / np.sqrt(x1**2 + x2**2)
    def b3(self, x1, x2, x3):
        return np.zeros_like(x1)

    def _curv_v(self):
        sim = self.sim
        p0 = sim["test_electrons"]["tracks"]["p1"][:,0]**2 + sim["test_electrons"]["tracks"]["p2"][:,0]**2 + sim["test_electrons"]["tracks"]["p3"][:,0]**2

        gamma_0 = np.sqrt(1 + p0)

        p_par0 = (sim["test_electrons"]["tracks"]["p1"][:,0] * self.b1(self.x1_0, self.x2_0, self.x3_0) + \
                sim["test_electrons"]["tracks"]["p2"][:,0] * self.b2(self.x1_0, self.x2_0, self.x3_0) + \
                sim["test_electrons"]["tracks"]["p3"][:,0] * self.b3(self.x1_0, self.x2_0, self.x3_0) ) / self.B0
        
        v_par = p_par0 / gamma_0
        vc = self.rqm * p_par0**2 / self.B0 / gamma_0 * self.direc / self.R0

        return vc, v_par

    def get_curv_traj(self, t):
        t = np.asarray(t)              # shape: (nt,)
        omega = self.v_par / self.R0      # shape: (npart,)

        x3 = self.x3_0[:, None] + np.outer(self.vc, t)

        phase = np.outer(omega, t) + self.phi0[:, None]

        x1 = self.R0[:, None] * np.cos(phase)
        x2 = self.R0[:, None] * np.sin(phase)

        return np.array([x1, x2, x3])
    

In [ ]:
test = "Curv"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim = createSimDic(path, sim_labels, test)

In [ ]:
t = sim["Gca"]["1000"]["test_electrons"]["tracks"]["t"][0,1]
traj_theo = curvDriftTheo(sim["Gca"]["1000"], B).get_curv_traj(t)[:,:,0]

grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

from matplotlib.lines import Line2D

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        err = (np.abs(traj - traj_theo)) / L

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    
    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("t = " + str(round(t * B / 2.0 / np.pi, 1)) + r" $[2\pi / \Omega_e]$")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}


sim = createSimDic(path, sim_labels, test)

In [ ]:
grid = sim["Gca"]["1000"]["test_electrons"]["tracks"].grid
grid = [float(grid[0, 0]), float(grid[0, 1])]
L = grid[1] - grid[0]

components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)
for pusher in sim.keys():
    X = []    
    Y = []
    YERR = []    
    YMax = []
    for dtw in sim[pusher].keys():
        t = sim[pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
        traj_theo = curvDriftTheo(sim[pusher][dtw], 1000).get_curv_traj(t)[:,:,0]

        x1 = sim[pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
        x2 = sim[pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
        x3 = sim[pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

        traj = np.array([x1, x2, x3])

        err = (np.abs(traj - traj_theo)) / L

        err_radial = np.sqrt(err[0]**2 + err[1]**2)

        err = np.array([err_radial, err[2]])

        mean = np.mean(err, axis=1)
        std = np.std(err, axis=1, ddof=1)

        X.append(float(dtw.replace("_", ".")))
        Y.append(mean)
        YERR.append(std)
        YMax.append(np.max(err, axis=1))

    if pusher == "gcaCorrNoBoris":
         linestyle = '--'
    else:        linestyle = '-'
    

    X = np.asarray(X, dtype=float)
    Y = np.asarray(Y, dtype=float)
    YERR = np.asarray(YERR, dtype=float)
    YMax = np.asarray(YMax, dtype=float)

    for i, ax in enumerate(axes):
        eb = ax.errorbar(
            X, Y[:, i], yerr=YERR[:, i],
            label=fr"{pusher}",
            fmt='o',
            markersize=5,
            linestyle=linestyle,
            linewidth=1.2,
            capsize=3,
            elinewidth=1.0,
        )

        ax.plot(
            X,
            YMax[:, i],
            marker='x',
            linestyle='none',
            markersize=5,
            markeredgewidth=1.0,
            zorder=3,
            color=eb.lines[0].get_color(),
        )
        ax.set_ylabel(fr"{components[i]} error / L")
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)

axes[0].set_title("1st step")
axes[0].legend()
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim = createSimDic(path, sim_labels, test)

test = "Curv_dx"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim2 = createSimDic(path, sim_labels, test)

sims = {"80": sim, "120": sim2}

In [ ]:


components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

pushers = []
for dx in sims.keys():
    for pusher in sims[dx].keys():
        if pusher not in pushers:
            pushers.append(pusher)

color_map = {pusher: f"C{i}" for i, pusher in enumerate(pushers)}
style_map = {"120": "--"}
for dx in sims.keys():
    linestyle = style_map.get(dx, '-')
    t = sims[dx]["Gca"]["1000"]["test_electrons"]["tracks"]["t"][0,1]
    traj_theo = curvDriftTheo(sims[dx]["Gca"]["1000"], B).get_curv_traj(t)[:,:,0]

    grid = sims[dx]["Gca"]["1000"]["test_electrons"]["tracks"].grid
    grid = [float(grid[0, 0]), float(grid[0, 1])]
    L = grid[1] - grid[0]

    for pusher in sims[dx].keys():
        X = []    
        Y = []
        YERR = []    
        YMax = []
        for dtw in sims[dx][pusher].keys():
            x1 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
            x2 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
            x3 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

            traj = np.array([x1, x2, x3])

            err = (np.abs(traj - traj_theo)) / L

            err_radial = np.sqrt(err[0]**2 + err[1]**2)

            err = np.array([err_radial, err[2]])

            mean = np.mean(err, axis=1)
            std = np.std(err, axis=1, ddof=1)

            X.append(float(dtw.replace("_", ".")))
            Y.append(mean)
            YERR.append(std)
            YMax.append(np.max(err, axis=1))
        
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)
        YERR = np.asarray(YERR, dtype=float)
        YMax = np.asarray(YMax, dtype=float)

        for i, ax in enumerate(axes):
            eb = ax.errorbar(
                X, Y[:, i], yerr=YERR[:, i],
                fmt='o',
                markersize=5,
                linestyle=linestyle,
                linewidth=1.2,
                capsize=3,
                elinewidth=1.0,
                color=color_map[pusher],
            )

            ax.plot(
                X,
                YMax[:, i],
                marker='x',
                linestyle='none',
                markersize=5,
                markeredgewidth=1.0,
                zorder=3,
                color=eb.lines[0].get_color(),
            )
            ax.set_ylabel(fr"{components[i]} error / L")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(True, alpha=0.3)

pusher_handles = [
    Line2D([0], [0], color=color_map[pusher], marker='o', linestyle='-', label=pusher)
    for pusher in pushers
]
style_handles = [
    Line2D([0], [0], color='black', linestyle=style_map.get(dx, '-'), label=fr"{dx} cells")
    for dx in sims.keys()
]

axes[0].set_title("t = " + str(round(t * B / 2.0 / np.pi, 1)) + r" $[2\pi / \Omega_e]$")
axes[0].legend(handles=pusher_handles + style_handles)
axes[-1].set_xlabel("dtw")
fig.tight_layout()


In [ ]:
test = "Curv_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim = createSimDic(path, sim_labels, test)

test = "Curv_dx_1step"
path = f"/home/exxxx5/Tese/Decks/StudyConvergence"
sim_labels = {
    'Boris': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'Gca': ["1000", "500", "100", "50", "10", "1", "0_1"],   
    'GcaCorr': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrV4': ["1000", "500", "100", "50", "10", "1", "0_1"],
    'gcaCorrV5': ["1000", "500", "100", "50", "10", "1", "0_1"],
    # 'gcaCorrNoBoris': ["1000", "500", "50", "100", "10", "1", "0_1"],
}

B = 1000.
sim2 = createSimDic(path, sim_labels, test)

sims = {"80": sim, "120": sim2}

In [ ]:
components = ["r", "x3"]
fig, axes = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

pushers = []
for dx in sims.keys():
    for pusher in sims[dx].keys():
        if pusher not in pushers:
            pushers.append(pusher)

color_map = {pusher: f"C{i}" for i, pusher in enumerate(pushers)}
style_map = {"120": "--"}
for dx in sims.keys():
    linestyle = style_map.get(dx, '-')

    for pusher in sims[dx].keys():
        X = []    
        Y = []
        YERR = []    
        YMax = []
        for dtw in sims[dx][pusher].keys():
            t = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["t"][0,1]
            traj_theo = curvDriftTheo(sims[dx][pusher][dtw], B).get_curv_traj(t)[:,:,0]

            grid = sims[dx][pusher][dtw]["test_electrons"]["tracks"].grid
            grid = [float(grid[0, 0]), float(grid[0, 1])]
            L = grid[1] - grid[0]

            x1 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x1"][:,1]
            x2 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x2"][:,1]
            x3 = sims[dx][pusher][dtw]["test_electrons"]["tracks"]["x3"][:,1]

            traj = np.array([x1, x2, x3])

            err = (np.abs(traj - traj_theo)) / L

            err_radial = np.sqrt(err[0]**2 + err[1]**2)

            err = np.array([err_radial, err[2]])

            mean = np.mean(err, axis=1)
            std = np.std(err, axis=1, ddof=1)

            X.append(float(dtw.replace("_", ".")))
            Y.append(mean)
            YERR.append(std)
            YMax.append(np.max(err, axis=1))
        
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)
        YERR = np.asarray(YERR, dtype=float)
        YMax = np.asarray(YMax, dtype=float)

        for i, ax in enumerate(axes):
            eb = ax.errorbar(
                X, Y[:, i], yerr=YERR[:, i],
                fmt='o',
                markersize=5,
                linestyle=linestyle,
                linewidth=1.2,
                capsize=3,
                elinewidth=1.0,
                color=color_map[pusher],
            )

            ax.plot(
                X,
                YMax[:, i],
                marker='x',
                linestyle='none',
                markersize=5,
                markeredgewidth=1.0,
                zorder=3,
                color=eb.lines[0].get_color(),
            )
            ax.set_ylabel(fr"{components[i]} error / L")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(True, alpha=0.3)

pusher_handles = [
    Line2D([0], [0], color=color_map[pusher], marker='o', linestyle='-', label=pusher)
    for pusher in pushers
]
style_handles = [
    Line2D([0], [0], color='black', linestyle=style_map.get(dx, '-'), label=fr"{dx} cells")
    for dx in sims.keys()
]

axes[0].set_title("1st step")
axes[0].legend(handles=pusher_handles + style_handles)
axes[-1].set_xlabel("dtw")
fig.tight_layout()
